# Imports

In [ ]:
import os
import glob
import random
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"
import math
from functools import partial
import numpy as np
import scipy as sp
import tqdm
import matplotlib.pyplot as plt
from pathlib import Path
from functools import partial
import datetime
import json
from PIL import Image
from attr import dataclass
import scipy.io as scio
import h5py
import torch
from torch import nn, Tensor, optim
import torch.nn.functional as F
from torchvision import datasets
from torchmetrics.image import MultiScaleStructuralSimilarityIndexMeasure, StructuralSimilarityIndexMeasure
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR, StepLR, SequentialLR, LinearLR, OneCycleLR
from torch.utils.data import  DataLoader
from torchvision.transforms import v2
from torchinfo import summary
import pandas as pd
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor, EarlyStopping, TQDMProgressBar
from pytorch_lightning.loggers import CSVLogger, TensorBoardLogger, MLFlowLogger
import pynop

from utils import plot_interactive, plot_comparison, get_simulation_by_id, animate_results_mpl

%matplotlib inline


# Model creation


In [ ]:
# Dictionary containing all model configurations
# number of past slice (ground truth) given to the network or latent representation the case of CausalLNO
steps = 6
input_channels = 2

models_config = {
    "ITLNO": {
        "model": pynop.ITLNO,
        "params": {
            "in_channels": input_channels * steps,
            "out_channels": input_channels,
            "modes": 16,
            "num_blocks": 4,
            "hidden_channels": 128,
            "num_heads": 4,
            "linear_kernel": False,
            "mlp_layers": 2,
            "mlp_dim": 128,
            "mlp_factor": 2,
            "activation": nn.GELU,
            "mlp_act": nn.GELU,
            "ndim": 2,
            "orthogonal_init": True,
            "rmsnorm": True,
            "basis_mode": "free",
            "verbose": True,
            "std_init": 1e-1,
            "pe": "random_ipe",
            "pe_param":10
        },
    },
    "LatentGalerkin": {
        "model": pynop.LatentGalerkin,
        "params": {
            "in_channels": input_channels * steps,
            "out_channels": input_channels,
            "modes": 256,
            "num_blocks": 4,
            "hidden_channels": 128,
            "num_heads": 4,
            "mlp_layers": 1,
            "mlp_dim": 128,
            "mlp_factor": 2,
            "activation": nn.GELU,
            "mlp_act": nn.GELU,
            "dim": 2,
            "std_init": 1e-1,
            "latent_attention": "vanilla",
            "galerkin_norm": True,
            "verbose": True,
            "pe": None,
            "pe_param": 10,
        },
    },
    "LatentTransolver": {
        "model": pynop.LatentTransolver,
        "params": {
            "in_channels": input_channels * steps,
            "out_channels": input_channels,
            "modes": 256,
            "num_blocks": 4,
            "hidden_channels": 128,
            "num_heads": 4,
            "mlp_layers": 1,
            "mlp_dim": 128,
            "mlp_factor": 2,
            "activation": nn.GELU,
            "mlp_act": nn.GELU,
            "std_init": 1e-1,
            "rmsnorm": True,
            "latent_attention": "vanilla"
        },
    },
    "Transolver_linear": {
        "model": pynop.Transolver,
        "params": {
            "in_ch": input_channels * steps,
            "out_ch": input_channels,
            "slice_num": 128,
            "n_layers": 4,
            "n_hidden": 128,
            "dropout": 0.05,
            "n_head": 4,
            "mode": "linear",
            "activation": nn.GELU,
            "mlp_ratio": 2,
            "dim": 2,
            "cond_dim": None,
        },
    },
    "Transolver3": {
        "model": pynop.Transolver,
        "params": {
            "in_ch": input_channels * steps,
            "out_ch": input_channels,
            "slice_num": 256,
            "n_layers": 4,
            "n_hidden": 128,
            "dropout": 0.05,
            "n_head": 4,
            "mode": "t3",
            "activation": nn.GELU,
            "mlp_ratio": 2,
            "dim": 2,
            "cond_dim": None,
        },
    },
    "LNO": {
        "model": pynop.LatentNO,
        "params": {
            "in_channels": input_channels * steps,
            "out_channels": input_channels,
            "d_model":  128,
            "modes":  256,
            "num_heads":  4,
            "num_blocks":  4,
            "mlp_layers":  1,
            "rmsnorm": True,
            "mlp_dim": 128,
            "dropout":  0,
            "transformer_mlp_factor": 2,
            "activation": nn.GELU,
            "dim": 2,
            "tau_ini":  0.1,
            "pe":"random_fourier",
            "pe_param":10,
            "cond_dim":None,
            "std_init":0.02,
        },
    },
    "FNOvanilla": {
        "model": pynop.FNO,
        "params": {
            "in_channels": input_channels * steps,
            "out_channels": input_channels,
            "modes": (32, 32),
            "hidden_channels": (20, 20, 20, 20),
            "blocks": ["FNO", "FNO", "FNO", "FNO"],
            "spectral_layer_type": "standard",
            "spectral_compression_factor": None,
            "trainable_pos_encoding": False,
            "fixed_pos_encoding": False,
            "trainable_pos_encoding_modes": (16, 16),
            "trainable_pos_encoding_dims": 8,
        },
    },
    "SwinNO": {
        "model": pynop.SwinNO,
        "params": {
            "in_channels": input_channels * steps,
            "out_channels": input_channels,
            "num_windows": 16,
            "latent_dim": 128,
            "num_blocks": 4,
            "num_heads": 4,
            "mlp_factor": 2,
            "activation": nn.GELU,
            "block_type": "swin",
            "bc_mode": "circular",
            "ndim":  2,
        },
    },
    "MSSwinNO": {
        "model": pynop.MultiscaleSWINNO,
        "params": {
            "in_channels": input_channels * steps,
            "out_channels": input_channels,
            "grid_resolutions":[(32,32), (16,16)],
            "enc_num_windows": [(8, 8), None],
            "dec_num_windows": [(8, 8), (8, 8)],
            "model_dim": [96, 128, 192],
            "num_heads": 4,
            "mlp_factor": 2,
            "activation": nn.GELU,
            "rmsnorm": True,
        },
    },
}


In [ ]:
# name = 'LatentTransolver'
# name = "Transolver_linear"
# name = 'FNO'
# name = "LatentGalerkin"
# name = "FNOvanilla"
# name = "ITLNO"
# name = "FNO_separable"
# name = "LNO"
name = "SwinNO"
# name = "Transolver3"
# name = "MSSwinNO"

# Instantiate model
if name == "CLNO" or name == "VAETimeAttentionLatentGalerkin":
    in_steps = 1
else:
    in_steps = steps

model = models_config[name]["model"](**models_config[name]["params"]).to("cuda")

# Model summary

In [ ]:
# summary(model, input_size=(1,2,128,128))
data = torch.randn((3, input_channels*in_steps, 128, 128)).to("cuda")
time = torch.rand(3).unsqueeze(-1).float().to("cuda")
print(summary(model, input_data=data, depth=5, col_names=("input_size", "output_size", "num_params")))
# print(summary(model, input_data=(data, time), depth=4, col_names=("input_size", "output_size", "num_params")))


# Data loading, creation of the train/val sets and dataloaders

In [ ]:
# datapath = Path("/media/jlux/SSD2/pdebench/2d_reaction_diffusion/133017.hdf5")
# stats=pynop.inspect_PDEBenchDataset(datapath, DT=5e-2)
# print(stats)
# With DT=5/100²
# ==================================================
#               PDEBENCH INSPECTION REPORT
# ==================================================
# Simulations : 1000
# Time steps  : 101
# Resolution  : (128, 128)
# Channels    : 2
# --------------------------------------------------
# CHANNEL 0:
#   [Values] Min: -5.2797 | Max: 5.5866
#   [Values] Mean: -0.0311 | Std: 0.1436
#   [Deriv ] Mean: -0.014908 | Std: 1.962596
# --------------------------------------------------
# CHANNEL 1:
#   [Values] Min: -5.3747 | Max: 5.1440
#   [Values] Mean: -0.0199 | Std: 0.1113
#   [Deriv ] Mean: -0.011219 | Std: 1.992754
# --------------------------------------------------
# ==================================================

In [ ]:
# datapath = Path("F:/Projets/2D_diff-react_NA_NA.h5")
datapath = Path("/media/jlux/SSD2/pdebench/2d_reaction_diffusion/133017.hdf5")
# datapath = Path("/home/jlux/Nextcloud/Projets/NBSINFRA/Fatima Shared Folder/data/simple_concentrations_with porosity.hdf5")

# stats du dataset
# 'global_min': array([-5.279711 , -5.3747244], dtype=float32),
# 'global_max': array([5.5866303, 5.1440244], dtype=float32)

# std = (0.1113 + 0.1436) * 0.5
# shift = [0.0311, 0.0199]  # -> values > 0
# scale = [1/0.1113, 1/0.1436] #/ std  # Division by the std to get std=1

shift=0
scale=1
unrolling = 10
window_size = unrolling + steps + 1 # +1 because we return only one tensor for inputs/targets
delta = 2
min_id = 1
max_id = 50
n_samples = 40
split_ratio=0.9

with h5py.File(datapath, "r") as f:
    all_ids = list(f.keys())

random.seed(42)
random.shuffle(all_ids)
if n_samples is not None:
    n_samples = min(len(all_ids), n_samples)
    all_sample_ids = all_ids[:n_samples]
else:
    all_sample_ids = all_ids

n_train = int(n_samples * split_ratio)
train_indexes = all_sample_ids[:n_train]
valid_indexes = all_sample_ids[n_train:]

print(train_indexes)
print(valid_indexes)

train_set = pynop.PDEBenchDataSet(
    datapath,
    T_unroll=window_size,
    step=delta,
    load_in_ram=True,
    indexes=train_indexes,
    min_id=min_id,
    max_id=max_id,
    shift=shift,
    scale=scale,
)
val_set = pynop.PDEBenchDataSet(
    datapath,
    T_unroll=window_size,
    step=delta,
    load_in_ram=True,
    indexes=valid_indexes,
    min_id=min_id,
    max_id=max_id,
    shift=shift,
    scale=scale,
)
print("\nTotal sequence length:", window_size)

# just to reshape the shift and scaling factor before inference!
def _prepare_transform(val):
    if isinstance(val, (list, np.ndarray, torch.Tensor)):
        tensor_val = torch.as_tensor(val).float()
        if tensor_val.ndim == 1:
            return tensor_val.view(1, -1, 1, 1)
        return tensor_val
    return val

In [ ]:
with h5py.File(datapath, "r") as f:
    print(list(f['0000'].keys()))

In [ ]:
batch_size = 4
train_dataloader = DataLoader(train_set, shuffle=True, batch_size=batch_size, num_workers=10)
valid_dataloader = DataLoader(
    val_set, shuffle=False, batch_size=batch_size, num_workers=10, persistent_workers=True, pin_memory=True
)

# Preparing the training
**Notes:**

- The metric RMSE, which is logged during training, is a mean over all spatial, channel and batch dimension. It is used here as it is often the metric given in the papers.

- Most of the  benchmarks use at least 10 previous time steps as input to predict the next time step.
- In Transolver, LNO and LinearNO, they mostly use Onecycle scheduler without curriculum training (full 10 timesteps rollout  at first epoch). 

In [ ]:
exp = "PDEBENCH/run1"  # root directory
weight_decay = 5e-4
max_epochs = 300
lr = 2e-4  # This is the maximum learning rate
start_monitoring = 200  # start saving the best models only after this epoch


train_config = pynop.TrainingConfig(
    dt=5e-2,
    start_autoregressive=0,  # epoch at which the AR steps starts increasing
    final_autoregressive=200,
    min_autoregressive_steps=2,  # number of AR steps before start_autoregressive
    max_autoregressive_steps=unrolling,
    detach_grad_steps=40,
    derivative_loss_weight=1.0,
    loss_fn=[pynop.DWMSELoss(), pynop.SpectralLoss(beta=2), pynop.SobolevLoss(l1=1.0, l2=1.0)],
    loss_weights=[1e4, 1e4, 1e4],
    noise_level=0,  # 1e-6,
    time_normalization=101,
    n_slices=steps,  # number of previous time steps used to predict the next one
    temporal_weighting=1.1,
)

scheduler_config = [
    {
        "scheduler": OneCycleLR,
        "max_lr": lr,
        "div_factor": 1e3,
        "final_div_factor": 1e2,
        "pct_start": 0.02,
        "interval": "step",
        "total_steps": max_epochs * len(train_dataloader),
        "frequency": 1,
    },
]

# scheduler_config = [
# {
#     "scheduler": ReduceLROnPlateau,
#     "mode": "min",
#     "patience": 5,
#     "factor": 0.5,
#     "monitor": "loss",
#     "interval": "epoch",
#     "frequency": 1,
#     "cooldown": 0,
# },
# ]

# scheduler_config = [
# {
#     "scheduler": pynop.cosine_schedule_with_warmup,
#     "num_warmup_steps": 5*len(train_dataloader),
#     "num_training_steps": max_epochs*len(train_dataloader),
#     "min_lr": 1e-6,
#     "num_cycles": 0.5,
#     "interval": "step",
#     "frequency": 1,
# },
# ]

# scheduler_config = [
#     {
#         "scheduler": StepLR,
#         "step_size": 100,
#         "gamma": 0.5,
#         "interval": "epoch",
#     },
# ]


print("Total number of steps:", max_epochs * len(train_dataloader))
model_name = name
loss_name = str(type(train_config.loss_fn)).split(".")[-1]
folder = f"{model_name}"  # _{loss_name}"
folder = Path(folder)
now = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
experiment = Path("/media/jlux/SSD2/ReactionDiffusion/") / Path(exp)
baselogdir = experiment / folder
logdir = Path(baselogdir / Path(f"{steps}steps_{now}"))
callbacks = []

loggers = [
    TensorBoardLogger(logdir / Path("tb_logs")),
    CSVLogger(logdir),
]

callbacks.append(
    ModelCheckpoint(
        monitor="val_RMSE", filename=os.path.join(logdir, "best_val_RMSE"), mode="min", save_top_k=2, save_last=False
    )
)

callbacks.append(
    pynop.CurriculumCheckpoint(
        start_epoch=start_monitoring,
        monitor="RMSE",
        filename=os.path.join(logdir, "best_RMSE"),
        mode="min",
        save_top_k=2,
        save_last=False,
    )
)
callbacks.append(
    pynop.CurriculumCheckpoint(
        start_epoch=start_monitoring,
        dirpath=logdir,
        filename="last",
        every_n_epochs=1,
        save_top_k=1,
        mode="max",
    )
)

callbacks.append(LearningRateMonitor(logging_interval="epoch"))
# callbacks.append(EarlyStopping(monitor="val_loss", min_delta=0.00, patience=10, verbose=False, mode="min"))


# print values in scientific format
class CustomProgressBar(TQDMProgressBar):
    def get_metrics(self, trainer, model):
        items = super().get_metrics(trainer, model)
        # On applique le format scientifique à tout le dictionnaire
        return {k: (f"{v:.3e}" if type(v) == float else v) for k, v in items.items()}


# callbacks.append(TQDMProgressBar(leave=True))
callbacks.append(CustomProgressBar(leave=True))

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=lr,
    weight_decay=weight_decay,
    betas=(0.9, 0.99),
)


lightning_model = pynop.MultiStepNOModel(
    model=model, train_config=train_config, optimizer=optimizer, scheduler_config=scheduler_config
)

# lightning_model.automatic_optimization = False

# checkpoint = torch.load(Path("/media/jlux/SSD2/ReactionDiffusion/exp5/ITLNO/with_sin_pe__10steps_20260510-135459") / Path("last.ckpt"))
# lightning_model.load_state_dict(checkpoint['state_dict'], strict=True)
# excluded_block = 'model.temporal_block'
# filtered_dict = {k: v for k, v in checkpoint["state_dict"].items() if not k.startswith(excluded_block)}
# lightning_model.load_state_dict(filtered_dict, strict=False)
# lightning_model.optimizer.load_state_dict(checkpoint['optimizer_states'][0])

torch.set_float32_matmul_precision("highest")
# torch.set_float32_matmul_precision("high")

trainer = pl.Trainer(
    max_epochs=max_epochs,
    callbacks=callbacks,
    accelerator="gpu",
    logger=loggers,
    num_sanity_val_steps=0,
    log_every_n_steps=50,
    gradient_clip_val=1,
    gradient_clip_algorithm="norm",
)

print("Log dir:", logdir)

# Tensorboard

In [ ]:
%load_ext tensorboard
# %tensorboard --logdir $baselogdir


# Training the model from scratch

In [ ]:
now = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

# lightning_model = torch.compile(lightning_model)

model.verbose=False
hyperparams = {"steps":steps,
              "shift":shift,
              "scale":scale,
              "lr0":lr,
              "batch_size":batch_size,
              "max_epoch":max_epochs,
              "train_idx":train_indexes,
              "valid_idx":valid_indexes}

logdir.mkdir(parents=True, exist_ok=True)
with open(logdir / f"{now}-config.json", "w") as f:
    json.dump(models_config[name], f, indent=4, default=str)
train_config.save(logdir / Path(f"{now}-train_config.json"))
with open(logdir / f"{now}-hyperparams.json", "w") as f:
    json.dump(hyperparams, f, indent=4, default=str)

trainer.fit(lightning_model, train_dataloaders=train_dataloader, val_dataloaders=valid_dataloader)


# Or resume from checkpoint

In [ ]:
# Train from checkpoint
trainer.fit(
    lightning_model,
    train_dataloaders=train_dataloader,
    val_dataloaders=valid_dataloader,
    ckpt_path="/media/jlux/SSD2/ReactionDiffusion/exp5/ITLNO/with_sin_pe__10steps_20260510-135459/last.ckpt",
)

# Evaluation

# Load model if needed

In [ ]:
logdir = Path("/media/jlux/SSD2/ReactionDiffusion/exp5/LatentGalerkinv2/with_sin_pe__10steps_20260505-205400")
checkpoint = torch.load(logdir / Path("last.ckpt"))
try:
    lightning_model.load_state_dict(checkpoint['state_dict'])
    model = lightning_model.model
except:
    state_dict = checkpoint['state_dict']
    state_dict_cleaned = {}
    for key, value in state_dict.items():
        # Le préfixe est généralement 'model.', mais on peut le vérifier pour être sûr
        if key.startswith("model."):
            new_key = key[6:]  # On retire le préfixe 'model.'
            state_dict_cleaned[new_key] = value
        else:
            # Si la clé n'a pas le préfixe, on la garde telle quelle
            state_dict_cleaned[key] = value
    model.load_state_dict(state_dict_cleaned)


# Multi-step model eval

In [ ]:
from utils import plot_interactive, plot_comparison, get_simulation_by_id

# Setup
device = "cuda" if torch.cuda.is_available() else "cpu"
datapath = Path("/media/jlux/SSD2/pdebench/2d_reaction_diffusion/133017.hdf5")
sample_id = train_set.sample_ids[random.randint(0,len(train_set.sample_ids))]
# sample_id = val_set.sample_ids[random.randint(0,len(val_set.sample_ids))]
full_inputs = get_simulation_by_id(datapath, sample_id).to("cuda")

# full_inputs = (full_inputs + _prepare_transform(shift).to(device)) * _prepare_transform(scale).to(device)
# full_inputs = (full_inputs + _prepare_transform(shift)) * _prepare_transform(scale)
full_inputs = full_inputs.unsqueeze(0).to(device)
print("Statistics of the trajectory:")
print(
    f"Max: {full_inputs.max().item()} \nMin: {full_inputs.min().item()} \n"
    f"Var: {full_inputs.var().item()} \nMean: {full_inputs.mean()}"
)

t_norm = 101
pred_list = []
RMSE_list = []
RMSE_over_std = []

memory = steps
print("memory", memory, "sample id:", sample_id)
print("Full input shape", full_inputs.shape)

start_idx = min_id
end_idx = full_inputs.shape[1] - 1

# Prepare initial input: take first 2 slices and add batch dim
# Shape: [1, N, C, H, W] (Batch, Sequence, Channels, Height, Width)
current_input = full_inputs[:, start_idx : start_idx + memory]
print(current_input.shape)
time_idx = torch.FloatTensor([start_idx + memory - 1])
time_idx = time_idx.unsqueeze(0).to(device)
model.to(device)
model.eval()
model.verbose = True
dt = 5e-2

with torch.no_grad():
    for target_idx in range(start_idx + memory, end_idx):  # idx of the target

        current_time = torch.Tensor([target_idx / t_norm]).unsqueeze(0).to("cuda")

        # Reshape to [Batch, 2*C, H, W] for the model
        B, S, C, H, W = current_input.shape
        model_input = current_input.reshape(B, S * C, H, W)

        # Predict next frame: returns [B, C, H, W]
        preds = model(model_input) * dt + current_input[:,-1, ...]
        RMSE_list.append(torch.sqrt(torch.mean((preds - full_inputs[:, target_idx, ...]) ** 2)).item())
        RMSE_over_std.append(RMSE_list[-1] / full_inputs[:, target_idx, ...].std().item())

        if len(RMSE_list) > 1:
            print(
                f"t={target_idx}, error {RMSE_list[-1]:.2e}, RMSE/std {RMSE_over_std[-1]:.2e}, delta err = {RMSE_list[-1]-RMSE_list[-2]:.2e}"
            )
        else:
            print(f"t={target_idx}, error {RMSE_list[-1]:.2e}, RMSE/std {RMSE_over_std[-1]:.2e} ")
        pred_list.append(preds.squeeze(0).cpu())  # Store on CPU to save VRAM

        # Rolling window
        current_input = torch.cat([current_input[:, 1:, ...], preds.unsqueeze(1)], dim=1)

plt.plot(RMSE_list)
plt.yscale("log")
targets = full_inputs[:, start_idx + memory :, ...]
print("Average RMSE:", np.mean(RMSE_list))
print("Average RMSE/STD:", np.mean(RMSE_over_std))
timesteps = np.arange(start_idx + memory, end_idx, 1)
data = {"Timestep:" "RMSE": RMSE_list, "RMSEoverSTD": RMSE_over_std}

df = pd.DataFrame(data)
df.to_csv(os.path.join(logdir, "RMSE.csv"), index=False, encoding="utf-8")

# Visualization

In [ ]:
p = torch.stack(pred_list).detach().cpu()
g = targets[0].detach().cpu()
plot_interactive(p, g, channel_idx=1)

# Dataset Satistics

In [ ]:
datapath = Path("/media/jlux/SSD2/pdebench/2d_reaction_diffusion/133017.hdf5")
sample_id = train_set.sample_ids[20]
full_inputs = get_simulation_by_id(datapath, sample_id).to("cuda")

print(full_inputs.shape)
print(full_inputs.max().item(), full_inputs.min().item())
print(scale, shift)
dt = 5/100.
full_inputs = (full_inputs + shift) * scale
delta = torch.abs(full_inputs[1:, ...] - full_inputs[:-1, ...]) / dt
delta_mean = torch.mean(delta, dim=(1,2,3))

mean_per_frame = torch.mean(full_inputs, dim=(1,2,3))
std_per_frame = torch.mean(full_inputs**2, dim=(1,2,3)) - mean_per_frame**2
std_total = delta.std()
# print(torch.sqrt(std_per_frame))
print(std_total.item())

delta_max = torch.amax(delta, dim=(1,2,3))
delta_min = torch.amin(delta, dim=(1,2,3))
print(delta_max)
print(delta_min)

vrange = torch.amax(full_inputs, dim=(1,2,3)) - torch.amin(full_inputs,dim=(1,2,3))
plt.plot(torch.sqrt(std_per_frame).cpu().numpy(), label='std')
# plt.plot(mean_per_frame.abs().cpu().numpy(), label='mean')
# plt.plot(vrange.abs().cpu().numpy(), label='range')
plt.plot(delta_mean.cpu().numpy(), label='mean')
plt.plot(delta_min.cpu().numpy(), label='min')
plt.plot(delta_max.cpu().numpy(), label='max')
plt.yscale("log")
plt.legend()
print(vrange.mean().item(), std_per_frame.mean().item(), mean_per_frame.mean().item())

# END